In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
system_prompt = """Act as a Senior Data Architect with extensive experience 
in designing stream data processing projects in both On-Premise and Google Cloud Platform."""

user_prompt = """
I am planning to design a process to parse streaming json data from Kafka to BigQuery. 
I need to perform a few complex transformations on the data before publishing in BigQuery.
What are different options available for me and analyse Pros and Cons of each option and finally provide a recommendation.
"""

In [4]:
messages = [
    {"role":"system", "content": system_prompt},
    {"role":"user", "content": user_prompt}
]

In [5]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)

In [6]:
print(response.choices[0].message.content)

Short answer / executive summary
- If you need low-ops, cloud-managed, highly scalable stream processing with rich stateful transforms (joins, windows, enrichment, side inputs) and good observability: use Apache Beam on Dataflow reading from Kafka (direct KafkaIO if network allows, or via Pub/Sub if you prefer a cloud-native ingress) and write to BigQuery using the BigQuery Storage Write API (or BigQueryIO with proper batching). This gives the best balance of developer productivity, correctness, and operational simplicity on GCP.
- If you must remain Kafka-native (on-prem or Confluent-first) and want everything in the Kafka ecosystem: use a stream processing engine that runs alongside Kafka — Flink or Kafka Streams/ksqlDB — and sink to BigQuery (prefer Flink for complex stateful processing). This works but increases operations and integration effort.
- For simple transformations and fastest time-to-market, use Kafka Connect + BigQuery sink connector (Confluent or community). Easy to op